In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
# 1. Load the dataset
df = pd.read_csv('homework_8.1.csv')

# 2. Estimate propensity scores using logistic regression (Z predicts X)
X_features = sm.add_constant(df['Z'])
logit_model = sm.Logit(df['X'], X_features).fit()
df['propensity_score'] = logit_model.predict(X_features)

# 3. Calculate inverse probability weights (1/P for X=1 and 1/(1-P) for X=0)
df['ipw'] = np.where(df['X'] == 1, 
                     1 / df['propensity_score'], 
                     1 / (1 - df['propensity_score']))

# 4. Estimate the ATE (weighted average of Y for X=1 minus weighted average of Y for X=0)
mu_1 = np.mean(df['Y'] * df['X'] * df['ipw'])
mu_0 = np.mean(df['Y'] * (1 - df['X']) * df['ipw'])
ate = mu_1 - mu_0

print(f"Calculated ATE: {ate:.2f}")

Optimization terminated successfully.
         Current function value: 0.604614
         Iterations 5
Calculated ATE: 2.24


In [3]:
# 1. Load the dataset
df = pd.read_csv('homework_8.1.csv')

# 2. Fit the logistic regression model (Z predicting X) with an intercept
X_features = sm.add_constant(df['Z'])
logit_model = sm.Logit(df['X'], X_features).fit()

# 3. Predict the propensity scores
df['propensity_score'] = logit_model.predict(X_features)

# 4. Display the first three values
print("First three propensity scores:")
print(df['propensity_score'].head(3).round(2).tolist())

Optimization terminated successfully.
         Current function value: 0.604614
         Iterations 5
First three propensity scores:
[0.84, 0.59, 0.71]


In [4]:
from scipy.spatial.distance import mahalanobis

# 1. Load the data
df = pd.read_csv('homework_8.2.csv')

# 2. Compute the 2x2 covariance matrix of Z1 and Z2, and its inverse
# As instructed, stack Z1 and Z2 into a 2xN matrix
Z_matrix = df[['Z1', 'Z2']].values.T
cov_matrix = np.cov(Z_matrix)
inv_cov = np.linalg.inv(cov_matrix)

# 3. Separate treated (X=1) and untreated (X=0) groups
treated = df[df['X'] == 1].reset_index(drop=True)
untreated = df[df['X'] == 0].reset_index(drop=True)

# 4. Perform nearest-neighbor matching with replacement
matched_untreated_Y = []
untreated_coords = untreated[['Z1', 'Z2']].values

for i in range(len(treated)):
    treated_coord = treated.loc[i, ['Z1', 'Z2']].values
    
    # Compute Mahalanobis distance to all untreated units
    distances = [mahalanobis(treated_coord, u, inv_cov) for u in untreated_coords]
    
    # Find the single nearest untreated item
    nearest_idx = np.argmin(distances)
    matched_untreated_Y.append(untreated.loc[nearest_idx, 'Y'])

# 5. Compute the average treatment effect across the matched pairs
ate_matched = np.mean(treated['Y']) - np.mean(matched_untreated_Y)
print(f"Calculated ATE: {ate_matched:.2f}")

Calculated ATE: 3.44


In [5]:
from scipy.spatial.distance import mahalanobis

# 1. Load the data
df = pd.read_csv('homework_8.2.csv')

# 2. Compute the inverse covariance matrix for Z1 and Z2
Z_matrix = df[['Z1', 'Z2']].values.T
cov_matrix = np.cov(Z_matrix)
inv_cov = np.linalg.inv(cov_matrix)

# 3. Get the coordinates of all untreated items
untreated_coords = df[df['X'] == 0][['Z1', 'Z2']].values

# 4. Check the options provided in the question
options = {
    'A': (0.2, -0.4),
    'B': (2.3, 1.2),
    'C': (1.5, -1.3),
    'D': (0.9, 1.4)
}

for opt, vals in options.items():
    # Calculate the Mahalanobis distance from this option to all untreated items
    distances = [mahalanobis(np.array(vals), u, inv_cov) for u in untreated_coords]
    print(f"Option {opt} {vals} -> Minimum Mahalanobis distance to untreated: {min(distances):.4f}")

Option A (0.2, -0.4) -> Minimum Mahalanobis distance to untreated: 0.0294
Option B (2.3, 1.2) -> Minimum Mahalanobis distance to untreated: 0.5056
Option C (1.5, -1.3) -> Minimum Mahalanobis distance to untreated: 0.0203
Option D (0.9, 1.4) -> Minimum Mahalanobis distance to untreated: 0.2322
